# EDA — 제구 성공률 예측 (control_success)

`data_description.md` 기준 데이터 탐색. 대회 규칙상 지켜야 할 것:
- `asof_*` 컬럼만 과거 이력 피처로 사용 가능 (미래/사후 정보 금지)
- **test.csv 행끼리 서로 참조하는 누적/빈도/target encoding 금지** — train.csv로만 그런 통계를 만들고 test엔 lookup만 해야 함
- `trackman_history.csv`는 train/test와 1:1 결합 테이블이 아님 (참고용 피처 소스)


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
SAMPLE_SUB_PATH = f"{DATA_DIR}/sample_submission.csv"
TRACKMAN_PATH = f"{DATA_DIR}/trackman_history.csv"

ID_COL = "row_id"
TARGET_COL = "control_success"


: 

## 1. 데이터 로드 및 기본 정보

In [2]:
train = pd.read_csv(TRAIN_PATH, encoding="utf-8-sig")
print(train.shape)
train.head()


(1475092, 49)


,row_id,season,game_month,game_dayofweek,inning,top_bottom,game_type,balls_before,strikes_before,outs_before,run_top_before,run_bot_before,run_total_before,score_diff_home,score_diff_pitcher_team,runner_on_1b,runner_on_2b,runner_on_3b,num_runners_on,base_state,home_win_expectancy,away_win_expectancy,li,pitcher_id,batter_id,pitcher_hand,batter_hand,pitcher_team_id,batter_team_id,asof_pitcher_n,asof_pitcher_success_rate,asof_pitcher_reverse_rate,asof_pitcher_middle_rate,asof_pitcher_ball_rate,asof_pitcher_strike_rate,asof_pitcher_prev1_game_success_rate,asof_pitcher_prev3_game_success_rate,asof_pitcher_prev5_game_success_rate,asof_pitcher_prev1_game_middle_rate,asof_pitcher_prev3_game_middle_rate,asof_pitcher_prev5_game_middle_rate,asof_batter_n,asof_batter_success_rate,asof_batter_middle_rate,asof_pitcher_pitchmix_n,asof_pitcher_fastball_rate,asof_pitcher_breaking_rate,asof_pitcher_offspeed_rate,control_success
0,TRAIN_0000001,2019,3,5,1,T,R,0,0,0,0,0,0,0,0,0,0,0,0,___,50.0,50.0,0.87,21961,21996,1,2,16,13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,NaN,NaN,0
1,TRAIN_0000002,2019,3,5,1,T,R,0,0,0,0,0,0,0,0,1,0,0,1,1__,46.5,53.5,1.44,21961,22103,1,1,16,13,1,0.00,1.000000,0.0,0.000000,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,1,1.0,0.0,0.0,0
2,TRAIN_0000003,2019,3,5,1,T,R,1,0,0,0,0,0,0,0,1,0,0,1,1__,46.5,53.5,1.44,21961,22103,1,1,16,13,2,0.00,0.500000,0.0,0.500000,0.00,NaN,NaN,NaN,NaN,NaN,NaN,1,0.0,0.0,2,1.0,0.0,0.0,0
3,TRAIN_0000004,2019,3,5,1,T,R,0,0,2,0,0,0,0,0,0,0,0,0,___,53.8,46.2,0.40,21961,23420,1,1,16,13,3,0.00,0.666667,0.0,0.333333,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,3,1.0,0.0,0.0,1
4,TRAIN_0000005,2019,3,5,1,T,R,0,1,2,0,0,0,0,0,0,0,0,0,___,53.8,46.2,0.40,21961,23420,1,1,16,13,4,0.25,0.500000,0.0,0.250000,0.25,NaN,NaN,NaN,NaN,NaN,NaN,1,1.0,0.0,4,1.0,0.0,0.0,1


In [3]:
train.dtypes


row_id                                   object
season                                    int64
game_month                                int64
game_dayofweek                            int64
inning                                    int64
top_bottom                               object
game_type                                object
balls_before                              int64
strikes_before                            int64
outs_before                               int64
run_top_before                            int64
run_bot_before                            int64
run_total_before                          int64
score_diff_home                           int64
score_diff_pitcher_team                   int64
runner_on_1b                              int64
runner_on_2b                              int64
runner_on_3b                              int64
num_runners_on                            int64
base_state                               object
home_win_expectancy                     

In [4]:
train.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
row_id,1475092,1475092,TRAIN_0000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
season,1475092.0,NaN,NaN,NaN,2021.528869,1.705827,2019.0,2020.0,2022.0,2023.0,2024.0
game_month,1475092.0,NaN,NaN,NaN,6.668617,1.927907,3.0,5.0,6.0,8.0,10.0
game_dayofweek,1475092.0,NaN,NaN,NaN,3.508963,1.732295,0.0,2.0,4.0,5.0,6.0
inning,1475092.0,NaN,NaN,NaN,4.965961,2.605404,1.0,3.0,5.0,7.0,13.0
top_bottom,1475092,2,T,752812,NaN,NaN,NaN,NaN,NaN,NaN,NaN
game_type,1475092,2,R,1314088,NaN,NaN,NaN,NaN,NaN,NaN,NaN
balls_before,1475092.0,NaN,NaN,NaN,0.898468,0.969742,0.0,0.0,1.0,2.0,3.0
strikes_before,1475092.0,NaN,NaN,NaN,0.871871,0.824605,0.0,0.0,1.0,2.0,2.0
outs_before,1475092.0,NaN,NaN,NaN,0.981432,0.816727,0.0,0.0,1.0,2.0,2.0


## 2. 타겟 분포 (`control_success`)

클래스 불균형 여부, 전체 성공률(베이스라인 추정) 확인.


In [5]:
target_counts = train[TARGET_COL].value_counts().sort_index()
target_ratio = train[TARGET_COL].mean()
print(target_counts)
print(f"\n전체 제구 성공률: {target_ratio:.4f}")


control_success
0    702489
1    772603
Name: count, dtype: int64

전체 제구 성공률: 0.5238


## 3. 결측치 확인

In [6]:
missing = train.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(2)
pd.DataFrame({"n_missing": missing, "pct": missing_pct})


,n_missing,pct
asof_pitcher_prev1_game_success_rate,29185,1.98
asof_pitcher_prev3_game_success_rate,29185,1.98
asof_pitcher_prev5_game_success_rate,29185,1.98
asof_pitcher_prev1_game_middle_rate,29185,1.98
asof_pitcher_prev3_game_middle_rate,29185,1.98
asof_pitcher_prev5_game_middle_rate,29185,1.98
asof_batter_success_rate,830,0.06
asof_batter_middle_rate,830,0.06
asof_pitcher_success_rate,792,0.05
asof_pitcher_reverse_rate,792,0.05


## 4. Cold-start 표본 체크

`asof_pitcher_n`, `asof_batter_n`이 작을수록(특히 0) rate 계열 컬럼이 결측이거나 신뢰도가 낮음.
전처리 시 smoothing/shrinkage 전략이 필요한 비율을 가늠.


In [7]:
for col in ["asof_pitcher_n", "asof_batter_n", "asof_pitcher_pitchmix_n"]:
    zero_ratio = (train[col] == 0).mean()
    print(f"{col}: 0인 비율 = {zero_ratio:.4f}, 분포 = {train[col].describe()[['min','25%','50%','75%','max']].to_dict()}")


asof_pitcher_n: 0인 비율 = 0.0005, 분포 = {'min': 0.0, '25%': 684.0, '50%': 1776.0, '75%': 3725.0, 'max': 15449.0}
asof_batter_n: 0인 비율 = 0.0006, 분포 = {'min': 0.0, '25%': 788.0, '50%': 2263.0, '75%': 5007.0, 'max': 13927.0}
asof_pitcher_pitchmix_n: 0인 비율 = 0.0005, 분포 = {'min': 0.0, '25%': 684.0, '50%': 1776.0, '75%': 3725.0, 'max': 15449.0}


## 5. 범주형 피처 분포

In [8]:
cat_cols = ["top_bottom", "game_type", "base_state", "pitcher_hand", "batter_hand"]
for c in cat_cols:
    print(f"--- {c} ---")
    print(train[c].value_counts(dropna=False))
    print()


--- top_bottom ---
top_bottom
T    752812
B    722280
Name: count, dtype: int64

--- game_type ---
game_type
R    1314088
F     161004
Name: count, dtype: int64

--- base_state ---
base_state
___    777248
1__    293724
12_    125831
_2_    113627
1_3     48839
123     48408
__3     35513
_23     31902
Name: count, dtype: int64

--- pitcher_hand ---
pitcher_hand
2    1093741
1     381351
Name: count, dtype: int64

--- batter_hand ---
batter_hand
2    779345
1    695747
Name: count, dtype: int64



## 6. 시즌/시간 구조

In [9]:
print(train["season"].value_counts().sort_index())
print()
print(train["game_month"].value_counts().sort_index())


season
2019    237413
2020    244087
2021    247088
2022    247472
2023    245525
2024    253507
Name: count, dtype: int64

game_month
3      25756
4     206638
5     255547
6     253454
7     177147
8     221570
9     227845
10    107135
Name: count, dtype: int64


## 7. 카테고리별 타겟(성공률) 비교

상황에 따라 제구 성공률이 어떻게 달라지는지 — 피처 유용성 1차 확인.


In [10]:
for c in ["base_state", "pitcher_hand", "batter_hand", "top_bottom", "balls_before", "strikes_before", "outs_before"]:
    g = train.groupby(c)[TARGET_COL].agg(["mean", "count"]).sort_values("mean")
    print(f"--- {c} 별 성공률 ---")
    print(g)
    print()


--- base_state 별 성공률 ---
                mean   count
base_state                  
123         0.516134   48408
12_         0.521318  125831
___         0.523047  777248
_2_         0.524866  113627
_23         0.524889   31902
1__         0.525255  293724
1_3         0.527898   48839
__3         0.536057   35513

--- pitcher_hand 별 성공률 ---
                  mean    count
pitcher_hand                   
1             0.516705   381351
2             0.526228  1093741

--- batter_hand 별 성공률 ---
                 mean   count
batter_hand                  
1            0.520939  695747
2            0.526289  779345

--- top_bottom 별 성공률 ---
                mean   count
top_bottom                  
T           0.523116  752812
B           0.524443  722280

--- balls_before 별 성공률 ---
                  mean   count
balls_before                  
3             0.502092  124023
2             0.520782  254237
1             0.525941  444780
0             0.527568  652052

--- strikes_before 별 성공률 

## 8. 수치형 피처와 타겟의 상관관계 (참고용, 선형 상관일 뿐)

In [11]:
numeric_cols = train.select_dtypes(include=[np.number]).columns.drop([TARGET_COL])
corr_with_target = train[numeric_cols].corrwith(train[TARGET_COL]).sort_values(key=np.abs, ascending=False)
corr_with_target


asof_pitcher_success_rate               0.084271
asof_pitcher_prev5_game_success_rate    0.081569
asof_pitcher_reverse_rate              -0.079493
asof_pitcher_prev3_game_success_rate    0.077776
asof_pitcher_prev1_game_success_rate    0.061689
asof_batter_success_rate                0.058946
season                                 -0.048272
asof_batter_n                          -0.036276
asof_pitcher_middle_rate               -0.036075
asof_batter_middle_rate                -0.033821
asof_pitcher_prev5_game_middle_rate    -0.031952
asof_pitcher_prev3_game_middle_rate    -0.028296
asof_pitcher_prev1_game_middle_rate    -0.019460
asof_pitcher_ball_rate                 -0.017735
asof_pitcher_offspeed_rate              0.013798
asof_pitcher_breaking_rate             -0.013618
inning                                 -0.013010
pitcher_id                             -0.012497
balls_before                           -0.012057
asof_pitcher_pitchmix_n                -0.011613
asof_pitcher_n      

## 9. 선수 ID 카디널리티

`pitcher_id`/`batter_id`가 얼마나 다양한지, 선수당 표본 수 분포 — as-of 피처의 신뢰도와 직결.


In [12]:
for c in ["pitcher_id", "batter_id"]:
    n_unique = train[c].nunique()
    per_id_count = train[c].value_counts()
    print(f"{c}: unique={n_unique}")
    print(per_id_count.describe())
    print()


pitcher_id: unique=792
count      792.000000
mean      1862.489899
std       2540.574358
min          2.000000
25%        173.500000
50%        856.500000
75%       2569.000000
max      15450.000000
Name: count, dtype: float64

batter_id: unique=830
count      830.000000
mean      1777.219277
std       2911.960032
min          1.000000
25%         87.250000
50%        473.000000
75%       2010.750000
max      13928.000000
Name: count, dtype: float64



## 10. `test.csv` / `sample_submission.csv` 형식 확인

배포본은 형식 확인용 5건 샘플 (`data_description.md` 참고).


In [13]:
test = pd.read_csv(TEST_PATH, encoding="utf-8-sig")
sample_sub = pd.read_csv(SAMPLE_SUB_PATH, encoding="utf-8-sig")
print(test.shape, sample_sub.shape)
display(test)
display(sample_sub)


(5, 48) (5, 2)


,row_id,season,game_month,game_dayofweek,inning,top_bottom,game_type,balls_before,strikes_before,outs_before,run_top_before,run_bot_before,run_total_before,score_diff_home,score_diff_pitcher_team,runner_on_1b,runner_on_2b,runner_on_3b,num_runners_on,base_state,home_win_expectancy,away_win_expectancy,li,pitcher_id,batter_id,pitcher_hand,batter_hand,pitcher_team_id,batter_team_id,asof_pitcher_n,asof_pitcher_success_rate,asof_pitcher_reverse_rate,asof_pitcher_middle_rate,asof_pitcher_ball_rate,asof_pitcher_strike_rate,asof_pitcher_prev1_game_success_rate,asof_pitcher_prev3_game_success_rate,asof_pitcher_prev5_game_success_rate,asof_pitcher_prev1_game_middle_rate,asof_pitcher_prev3_game_middle_rate,asof_pitcher_prev5_game_middle_rate,asof_batter_n,asof_batter_success_rate,asof_batter_middle_rate,asof_pitcher_pitchmix_n,asof_pitcher_fastball_rate,asof_pitcher_breaking_rate,asof_pitcher_offspeed_rate
0,TEST_000001,2025,7,1,4,T,R,1,0,2,2,0,2,-2,-2,0,0,0,0,___,29.7,70.3,0.34,21813,23143,2,2,16,21,3465,0.489466,0.239250,0.144012,0.380664,0.418470,0.350877,0.327103,0.335821,0.157895,0.158879,0.171642,1999,0.506753,0.148574,3465,0.483117,0.369120,0.147763
1,TEST_000017,2025,3,2,4,B,R,2,0,1,7,4,11,-3,3,1,0,0,1,1__,21.4,78.6,1.29,24745,22026,1,2,14,16,86,0.290698,0.395349,0.209302,0.500000,0.348837,NaN,NaN,NaN,NaN,NaN,NaN,11153,0.520308,0.153412,86,0.534884,0.383721,0.081395
2,TEST_000213,2025,6,3,1,B,R,0,0,2,1,0,1,-1,1,1,0,0,1,1__,41.8,58.2,0.83,24198,24790,2,2,19,14,486,0.475309,0.273663,0.201646,0.380658,0.419753,0.440860,0.495902,0.479365,0.225806,0.209016,0.200000,0,NaN,NaN,486,0.427984,0.230453,0.341564
3,TEST_005332,2025,3,5,1,B,R,0,0,0,0,0,0,0,0,0,0,0,0,___,54.8,45.2,0.87,24713,23646,1,2,19,16,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12645,0.519968,0.158007,0,NaN,NaN,NaN
4,TEST_035185,2025,3,5,2,B,R,0,0,0,0,0,0,0,0,0,0,0,0,___,55.1,44.9,0.92,24713,24742,1,2,19,16,9,0.444444,0.222222,0.333333,0.333333,0.444444,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,9,0.666667,0.333333,0.000000


,row_id,control_success
0,TEST_000001,0.5
1,TEST_000017,0.5
2,TEST_000213,0.5
3,TEST_005332,0.5
4,TEST_035185,0.5


In [14]:
# train 피처 컬럼과 test 컬럼이 (타겟 제외) 동일한지 확인
train_feature_cols = set(train.columns) - {TARGET_COL}
test_cols = set(test.columns)
print("train에만 있는 컬럼:", train_feature_cols - test_cols)
print("test에만 있는 컬럼:", test_cols - train_feature_cols)


train에만 있는 컬럼: set()
test에만 있는 컬럼: set()


## 11. `trackman_history.csv` 개요

train/test와 1:1 결합 테이블이 아니므로, 여기서는 구조만 확인. 실제 활용은 투수/구종 단위 요약 피처를 만들어 `asof_*`처럼 시간 기준을 지켜서 붙이는 방식으로 설계해야 함.


In [15]:
trackman = pd.read_csv(TRACKMAN_PATH, encoding="utf-8-sig")
print(trackman.shape)
print(trackman["season"].value_counts().sort_index())
trackman.head()


(1793078, 30)
season
2019    255957
2020    279126
2021    301032
2022    307637
2023    315100
2024    334226
Name: count, dtype: int64


,trackman_id,season,game_date,game_month,game_dayofweek,trackman_game_id,pitch_no,inning,top_bottom,balls_before,strikes_before,outs_before,pitch_of_pa,pitcher_trackman_id,batter_trackman_id,pitcher_hand,batter_hand,pitcher_team,batter_team,tagged_pitch_type,auto_pitch_type,pitch_type_group,rel_speed,spin_rate,induced_vert_break,horz_break,extension,rel_height,rel_side,zone_speed
0,1,2019,03/29/2019,3,4,20190329-Gocheok-1,113,4,Top,0,1,1,2,502010,75151,Right,Right,KIW_HER,SK_WYV,Fastball,Fastball,fastball,146.968,2347.37,47.1658,34.2972,1.64349,1.86123,0.768382,133.184
1,2,2019,03/29/2019,3,4,20190329-Gocheok-1,53,2,Top,0,1,0,2,502010,76812,Right,Right,KIW_HER,SK_WYV,Fastball,Fastball,fastball,148.031,2211.66,49.9042,38.0496,1.58619,1.92934,0.729473,135.047
2,3,2019,03/29/2019,3,4,20190329-Gocheok-1,14,1,Top,1,1,1,3,502010,457449,Right,Right,KIW_HER,SK_WYV,Fastball,Fastball,fastball,146.966,2248.03,48.6507,30.0408,1.62189,1.89933,0.706493,131.781
3,4,2019,03/29/2019,3,4,20190329-Gocheok-1,81,3,Top,0,0,0,1,502010,76802,Right,Right,KIW_HER,SK_WYV,Fastball,Fastball,fastball,145.275,2211.69,48.4108,34.2192,1.58921,1.90932,0.796656,133.251
4,5,2019,03/29/2019,3,4,20190329-Gocheok-1,179,6,Top,0,0,1,1,502010,76802,Right,Right,KIW_HER,SK_WYV,Cutter,Slider,breaking,138.768,2441.32,21.5144,18.4652,1.75681,1.83312,0.781440,127.422


In [16]:
trackman.isna().mean().sort_values(ascending=False).head(15)


spin_rate             0.006952
horz_break            0.005763
induced_vert_break    0.005612
zone_speed            0.004418
extension             0.004303
rel_side              0.004249
rel_height            0.004248
rel_speed             0.004248
auto_pitch_type       0.000040
season                0.000000
pitch_type_group      0.000000
tagged_pitch_type     0.000000
batter_team           0.000000
pitcher_team          0.000000
batter_hand           0.000000
dtype: float64

## 12. 다음 스텝 메모

- (여기에 관찰한 내용 바탕으로 피처 엔지니어링/모델링 방향 정리)
